# DBSCAN Clustering
**Dataset:** Synthetic Clusters (750 samples, 8 features)

DBSCAN groups points based on density — it automatically finds the number of clusters and labels outliers as noise (label = -1).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score

df = pd.read_csv('../data/synthetic_clusters.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
X = df.values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Scaled shape:', X_scaled.shape)

In [ ]:
# k-NN Distance Plot to find optimal epsilon
nbrs = NearestNeighbors(n_neighbors=5).fit(X_scaled)
distances, _ = nbrs.kneighbors(X_scaled)
k_dist = np.sort(distances[:, -1])[::-1]

plt.figure(figsize=(9, 4))
plt.plot(k_dist, color='steelblue', linewidth=1.5)
plt.axhline(0.5, color='red', linestyle='--', label='ε = 0.5')
plt.xlabel('Points (sorted by distance desc)')
plt.ylabel('5th Nearest Neighbor Distance')
plt.title('k-NN Distance Plot — Find Optimal ε at the Elbow')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Run DBSCAN
db = DBSCAN(
    eps=0.5,
    min_samples=5,
    metric='euclidean',
    algorithm='auto'
)
labels = db.fit_predict(X_scaled)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise    = (labels == -1).sum()
print(f'Clusters: {n_clusters}  |  Noise points: {n_noise} ({n_noise/len(labels)*100:.1f}%)')

In [ ]:
# Evaluate
valid = labels != -1
if n_clusters >= 2 and valid.sum() > n_clusters:
    sil = silhouette_score(X_scaled[valid], labels[valid])
    dbi = davies_bouldin_score(X_scaled[valid], labels[valid])
    print(f'Silhouette Score : {sil:.4f}  (higher = better)')
    print(f'Davies-Bouldin   : {dbi:.4f}  (lower  = better)')
else:
    print('Not enough clusters for metrics.')

In [ ]:
# Scatter plot (feature_1 vs feature_2)
unique_labels = sorted(set(labels))
colors = plt.cm.tab10(np.linspace(0, 1, max(len(unique_labels), 1)))

plt.figure(figsize=(9, 6))
patches = []
for i, lbl in enumerate(unique_labels):
    mask = labels == lbl
    clr  = 'lightgray' if lbl == -1 else colors[i]
    name = 'Noise' if lbl == -1 else f'Cluster {lbl}'
    plt.scatter(X_scaled[mask, 0], X_scaled[mask, 1],
                c=[clr], s=20, alpha=0.75, edgecolors='k', linewidths=0.2)
    patches.append(mpatches.Patch(color=clr, label=name))

plt.xlabel('feature_1 (scaled)')
plt.ylabel('feature_2 (scaled)')
plt.title(f'DBSCAN — {n_clusters} clusters · {n_noise} noise points')
plt.legend(handles=patches, loc='best', fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
# Epsilon sweep
eps_values = [0.2, 0.4, 0.6, 0.9, 1.2]
results = []
for e in eps_values:
    lb = DBSCAN(eps=e, min_samples=5).fit_predict(X_scaled)
    nc = len(set(lb)) - (1 if -1 in lb else 0)
    nn = (lb == -1).sum()
    results.append({'eps': e, 'clusters': nc, 'noise': nn})

res_df = pd.DataFrame(results)
print(res_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(res_df['eps'], res_df['clusters'], 'o-', color='steelblue')
axes[0].set_xlabel('ε'); axes[0].set_ylabel('Clusters'); axes[0].set_title('ε vs Clusters')
axes[1].plot(res_df['eps'], res_df['noise'], 'o-', color='tomato')
axes[1].set_xlabel('ε'); axes[1].set_ylabel('Noise Points'); axes[1].set_title('ε vs Noise')
plt.tight_layout(); plt.show()
print('Done!')